# Cursul 7. Variabile derivate și model
**Fluxuri de lucru pentru știința datelor · Sociologie · 2026**

Analiza exploratorie (C6) a găsit trei tipare: SCI scade cu distanța, este mai mare peste o graniță comună și crește cu migrația. Tiparele se suprapun: țările vecine sunt și apropiate, și au mai mulți migranți între ele. Un **model** le include simultan și estimează ce rămâne din fiecare când celelalte sunt ținute constante.

Înainte de model, variabilele se pregătesc (**ingineria variabilelor**, engl. *feature engineering*):

| Etapa | Ce face | Pe datele SCI |
|---|---|---|
| 1. Construcția | variabile noi din cele existente | rata migranților, distanța dintre țări, granița comună |
| 2. Transformarea | forma distribuției | logaritmul |
| 3. Scalarea | aceeași scară pentru toate | scorul z |
| 4. Extragerea | mai mulți indicatori într-unul | gruparea regiunilor (k-means) |
| 5. Selecția | setul final, fără redundanță | corelațiile predictorilor, VIF |

Apoi modelul: patru modele, interpretarea coeficienților, reziduurile. Un model pe date observaționale descrie **asocieri**, nu cauze.

## 0. Pregătirea
Bibliotecile, datele din C5 și distanța dintre centrele regiunilor (ca în C6). Svalbard se exclude (decizia din C6: numitor foarte mic).

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from sklearn.cluster import KMeans
from sqlalchemy import create_engine

sns.set_theme(style="whitegrid")
engine = create_engine("sqlite:///data/dsw.db")    # baza din C5

regiuni = pd.read_sql("select * from regiuni", engine)
perechi = pd.read_sql("select * from perechi_regiuni", engine)
tari = pd.read_sql("select * from perechi_tari", engine)

harta = gpd.read_file("https://gisco-services.ec.europa.eu/distribution/v2/nuts/geojson/NUTS_RG_20M_2024_3035_LEVL_2.geojson")               # contururile regiunilor, ca în C3
harta = harta.rename(columns={"NUTS_ID": "regiune"})
harta = harta[["regiune", "geometry"]].merge(regiuni, on="regiune")

In [ ]:
# Svalbard se exclude (C6)
perechi = perechi[(perechi["regiune_1"] != "NO0B") & (perechi["regiune_2"] != "NO0B")]

# distanța dintre centrele celor două regiuni, în kilometri (C6)
centre = harta.set_index("regiune").centroid
perechi["distanta_km"] = centre[perechi["regiune_1"]].values.distance(centre[perechi["regiune_2"]].values) / 1000

## 1. Construcția: variabile noi din cele existente
Modelul se face pe **perechi de țări**, pentru că migrația există doar între țări. Tabelul `tari` are SCI-ul și migranții; îi lipsesc trei variabile: rata migranților, distanța dintre țări și granița comună.

### 1.1. Rata migranților
Ca în C6: numărul de migranți depinde de mărimea țărilor, deci se împarte la populația celor două țări (la 1.000 de locuitori).

In [ ]:
# populația fiecărei țări în 2024 (Banca Mondială, C2)
bm = pd.read_csv("data/raw/bm.csv", keep_default_na=False, na_values=[""])   # NA = Namibia
pop = bm[(bm["indicator"] == "populatie") & (bm["an"] == 2024)].set_index("tara")["valoare"]

tari = tari.dropna(subset=["migranti"]).copy()           # doar perechile cu migrația cunoscută
tari["la_mie"] = 1000 * tari["migranti"] / (tari["tara_1"].map(pop) + tari["tara_2"].map(pop))
len(tari)

### 1.2. Distanța dintre două țări
Distanța dintre două țări se construiește din distanțele dintre regiunile lor: **mediana** distanțelor dintre toate perechile de regiuni ale celor două țări. Mediana nu e trasă de regiunile îndepărtate (insulele, teritoriile de peste mări).

O pereche de țări poate apărea ca „AT, RO” într-un tabel și ca „RO, AT” în celălalt; o **cheie** cu codurile în ordine alfabetică le face la fel.

In [ ]:
perechi["cheie"] = ["-".join(sorted(p)) for p in zip(perechi["tara_1"], perechi["tara_2"])]
tari["cheie"] = ["-".join(sorted(p)) for p in zip(tari["tara_1"], tari["tara_2"])]

distanta_tari = perechi.groupby("cheie")["distanta_km"].median()     # o distanță pe pereche de țări
tari["distanta_km"] = tari["cheie"].map(distanta_tari)
tari[["tara_1", "tara_2", "distanta_km"]].head()

### 1.3. Granița comună
Regiunile se lipesc în țări (`dissolve`), fiecare țară se lărgește cu 10 km (`buffer`), apoi se caută țările care se ating (`sjoin`). Cei 10 km acoperă micile goluri dintre contururile simplificate. Granița este o variabilă **binară**: 1 pentru țările vecine, 0 în rest.

In [ ]:
contur = harta.dissolve("tara")[["geometry"]]                         # o formă pe țară
largit = contur.buffer(10_000).to_frame("geometry")                    # lărgită cu 10 km
atingeri = gpd.sjoin(largit, contur, predicate="intersects")           # cu cine se atinge

vecini = set(zip(atingeri.index, atingeri["tara_right"]))
tari["granita"] = [int((a, b) in vecini) for a, b in zip(tari["tara_1"], tari["tara_2"])]
tari[tari["granita"] == 1][["tara_1", "tara_2"]].head(10)

**Exercițiu.** Câte perechi de țări vecine are tabelul? Două-trei dintre ele se verifică pe o hartă.

In [ ]:
# Suma coloanei granita (0 sau 1): câte perechi au valoarea 1

## 2. Transformarea: logaritmul
SCI, distanța și rata migranților au coadă lungă spre dreapta (C6). Logaritmul le aduce aproape de simetrie și transformă relațiile multiplicative în relații liniare. Granița (0 sau 1) nu se logaritmează: nu are coadă, iar log 0 nu există.

In [ ]:
for coloana in ["sci", "distanta_km", "la_mie"]:
    print(coloana, "asimetria:", round(tari[coloana].skew(), 2), "->", round(np.log10(tari[coloana]).skew(), 2))

In [ ]:
tari["log_sci"] = np.log10(tari["sci"])
tari["log_distanta"] = np.log10(tari["distanta_km"])
tari["log_migranti"] = np.log10(tari["migranti"])
tari["log_la_mie"] = np.log10(tari["la_mie"])

In [ ]:
sns.regplot(tari, x="log_distanta", y="log_sci");          # pe log-log, relația e aproape o dreaptă

**Același eșantion pentru toate modelele.** Modelele se compară doar pe exact aceleași perechi. Perechile cu o valoare lipsă se scot o singură dată, aici.

In [ ]:
date = tari.dropna(subset=["log_sci", "log_distanta", "log_la_mie", "granita"]).copy()
len(date)

**Exercițiu.** Același grafic pe valorile brute, fără logaritm. Se mai vede o dreaptă?

In [ ]:
# sns.regplot pe coloanele distanta_km și sci, fără logaritm

## 3. Scalarea și extragerea
### 3.1. Scorul z
Distanța (log) merge de la aproximativ 2 la 3,6, rata migranților (log) de la −3 la 1: scări diferite. **Scorul z** aduce fiecare variabilă la media 0 și abaterea standard 1: z = (x − media) / abaterea standard. Forma distribuției nu se schimbă, doar unitatea. Cu toate variabilele în scor z, coeficienții modelului se pot compara între ei (secțiunea 6).

In [ ]:
def scor_z(x):
    return (x - x.mean()) / x.std()

for coloana in ["log_sci", "log_distanta", "log_la_mie"]:
    date["z_" + coloana] = scor_z(date[coloana])

date[["log_distanta", "z_log_distanta", "log_la_mie", "z_log_la_mie"]].describe().round(2)

**Exercițiu.** Asimetria ratei migranților înainte și după scorul z. Scalarea schimbă forma?

In [ ]:
# skew() pe log_la_mie și pe z_log_la_mie, separate prin virgulă

### 3.2. Extragerea: gruparea regiunilor (clustering)
**Extragerea** rezumă mai mulți indicatori într-o singură variabilă. Analiza componentelor principale (PCA) dă un scor numeric; **gruparea** (engl. *cluster analysis*) dă o categorie: regiunile asemănătoare pe toți indicatorii ajung în aceeași grupă. Rezultatul este o variabilă derivată nouă, „tipul de regiune”, utilă unui proiect pe regiuni.

**k-means** împarte regiunile în k grupe: fiecare regiune merge la cel mai apropiat centru, apoi fiecare centru se mută în media regiunilor lui, până nu se mai schimbă nimic. Distanța se calculează pe toți indicatorii deodată, deci indicatorii trebuie să aibă aceeași scară: **scorul z este obligatoriu**. Altfel, PIB-ul (zeci de mii de euro) ar decide singur grupele.

In [ ]:
indicatori = ["pib_locuitor", "studii_superioare", "somaj", "varsta_medie", "log_densitate"]

reg = regiuni.copy()
reg["log_densitate"] = np.log10(reg["densitate"])          # coadă lungă: logaritmul (secțiunea 2)
reg = reg.dropna(subset=indicatori)                         # doar regiunile cu toți indicatorii
z = (reg[indicatori] - reg[indicatori].mean()) / reg[indicatori].std()   # scorul z
len(reg)

Câte grupe? **Inerția** este suma distanțelor (la pătrat) de la fiecare regiune la centrul grupei ei. Scade mereu când k crește; se caută punctul după care scade mult mai încet (**cotul**).

In [ ]:
inertie = [KMeans(n_clusters=k, n_init=10, random_state=1).fit(z).inertia_ for k in range(1, 9)]
plt.plot(range(1, 9), inertie, marker="o")
plt.xlabel("numărul de grupe, k"); plt.ylabel("inerția");

In [ ]:
grupare = KMeans(n_clusters=4, n_init=10, random_state=1).fit(z)
reg["grupa"] = grupare.labels_

# profilul fiecărei grupe: mediana indicatorilor, pe scara inițială
reg.groupby("grupa")[["pib_locuitor", "studii_superioare", "somaj", "varsta_medie", "densitate"]].median().round(1)

In [ ]:
reg["grupa"].value_counts()                                 # câte regiuni are fiecare grupă

In [ ]:
harta["grupa"] = harta["regiune"].map(reg.set_index("regiune")["grupa"])
ax = harta.plot(column="grupa", categorical=True, cmap="Set2", legend=True, figsize=(9, 7),
                missing_kwds={"color": "#eeeeee"})
ax.set(xlim=(2.5e6, 7.4e6), ylim=(1.4e6, 5.5e6), title="Patru tipuri de regiuni (k-means pe cinci indicatori)")
ax.set_axis_off()

Inerția nu are un cot foarte clar: k = 4 este o alegere, justificată prin grupe care se pot descrie. Numerele grupelor sunt doar etichete și pot ieși în altă ordine; grupele se numesc după profil. Din 298 de regiuni intră 266: celelalte nu au toți indicatorii (Elveția, Norvegia, Balcanii de Vest).

| Tipul de regiune | Regiuni | PIB/loc. | Studii superioare (%) | Șomaj (%) | Vârsta medie | Densitate | Exemple |
|---|---|---|---|---|---|---|---|
| metropolitane | 36 | 60.100 | 50,0 | 4,7 | 41,4 | 740 | capitalele: Praga, Bratislava, Budapesta, Varșovia, București-Ilfov, Madrid, Stockholm |
| dezvoltate | 119 | 38.300 | 34,6 | 3,7 | 45,1 | 139 | Germania, Austria, nordul Italiei, Polonia vestică |
| periferice, îmbătrânite | 90 | 27.350 | 27,9 | 7,7 | 45,8 | 65 | Grecia, sudul Spaniei și al Italiei, Franța rurală, România fără București, Bulgaria |
| tinere, cu șomaj ridicat | 21 | 20.500 | 22,8 | 10,3 | 34,2 | 140 | Turcia, teritoriile franceze de peste mări, Ceuta |

România se împarte: București-Ilfov este în grupa metropolelor, celelalte șapte regiuni în grupa periferiilor. Gruparea nu știe nimic despre țări sau hartă, dar capitalele ies împreună. Ca la comunitățile din C6, grupele arată *că* regiunile se aseamănă, nu *de ce*.

**De încercat.** Aceeași grupare **fără** scorul z: `KMeans(n_clusters=4, n_init=10, random_state=1).fit(reg[indicatori])`, apoi profilul grupelor. Ce indicator decide acum grupele? Apoi `n_clusters=3` și `n_clusters=6`.

## 4. Selecția: redundanța predictorilor
Doi predictori care spun aproape același lucru nu se pot separa în model: coeficienții lor devin instabili (**multicoliniaritate**). Prima verificare: corelațiile dintre predictori.

In [ ]:
predictori = ["log_distanta", "log_la_mie", "granita"]
date[predictori].corr().round(2)

A doua: **VIF** (factorul de inflație a varianței). Fiecare predictor se explică prin ceilalți; dacă R² e mare, predictorul e redundant. VIF = 1 / (1 − R²). Sub 5: acceptabil; între 5 și 10: atenție; peste 10: unul dintre predictori se scoate.

In [ ]:
for v in predictori:
    ceilalti = " + ".join(x for x in predictori if x != v)
    r2 = smf.ols(f"{v} ~ {ceilalti}", data=date).fit().rsquared      # cât de bine îl explică ceilalți
    print(v, "VIF =", round(1 / (1 - r2), 2))

Toate valorile sunt sub 2. Distanța și granița sunt legate (r = −0,63: țările vecine sunt apropiate), dar nu atât încât să nu se poată separa. Toți trei predictorii rămân.

## 5. Modelul
### 5.1. O dreaptă: tipar și reziduu
Un **model liniar** scrie fiecare valoare ca tipar plus reziduu: log SCI = nivel + pantă × log distanță + reziduu. **Metoda celor mai mici pătrate** alege nivelul și panta care fac suma pătratelor reziduurilor cât mai mică. **R²** este partea din variația lui log SCI explicată de model, între 0 și 1.

In [ ]:
m1 = smf.ols("log_sci ~ log_distanta", data=date).fit()
m1.params, m1.rsquared

### 5.2. Patru modele, câte o variabilă în plus
| Modelul | Ce conține |
|---|---|
| M1 | distanța |
| M2 | distanța și **numărul** de migranți |
| M3 | distanța și **rata** migranților |
| M4 | M3 și granița |

In [ ]:
m2 = smf.ols("log_sci ~ log_distanta + log_migranti", data=date).fit()
m3 = smf.ols("log_sci ~ log_distanta + log_la_mie", data=date).fit()
m4 = smf.ols("log_sci ~ log_distanta + log_la_mie + granita", data=date).fit()

modele = pd.DataFrame({"M1": m1.params, "M2": m2.params, "M3": m3.params, "M4": m4.params})
modele.loc["R²"] = [m1.rsquared, m2.rsquared, m3.rsquared, m4.rsquared]
modele.round(2)

- Distanța are semn negativ în toate modelele. Coeficientul ei scade de la −0,76 (M1) la −0,30 (M4): o parte din „efectul distanței” din M1 era migrație, pentru că țările apropiate au mai mulți migranți între ele.
- Cu numărul de migranți, R² crește de la 0,16 la 0,25; cu rata, la 0,53. Aceeași informație, construită altfel, explică de două ori mai mult: acesta este rostul variabilelor derivate.
- Granița adaugă foarte puțin peste distanță și migrație: R² rămâne 0,53.

### 5.3. Care variabilă contează mai mult?
Coeficienții din M4 au unități diferite. Pe scorurile z (secțiunea 3), fiecare coeficient spune cu câte abateri standard se schimbă log SCI când predictorul crește cu o abatere standard, deci se pot compara.

In [ ]:
m4z = smf.ols("z_log_sci ~ z_log_distanta + z_log_la_mie + granita", data=date).fit()
m4z.params.round(2)

Rata migranților are coeficientul cel mai mare (0,62), de aproape patru ori mai mare decât distanța (−0,16). Între perechile de țări, migrația contează mai mult decât distanța, odată ce amândouă sunt în model. Granița rămâne pe scara 0/1: coeficientul ei (0,22) este diferența dintre țările vecine și celelalte, în abateri standard ale lui log SCI.

## 6. Rezultatul complet și interpretarea
### 6.1. Ce conține `summary()`
`m4.summary()` dă tot rezultatul modelului, în trei blocuri: modelul în ansamblu (sus), coeficienții (mijloc), diagnosticul reziduurilor (jos).

In [ ]:
m4.summary()

**Sus: modelul în ansamblu.**
- `No. Observations: 263`, `Df Model: 3` (trei predictori), `Df Residuals: 259` (263 − 3 − 1, pentru constantă).
- `R-squared: 0.530`: modelul explică 53% din variația lui log SCI. `Adj. R-squared: 0.525`: același R², penalizat pentru numărul de predictori. R² crește la orice predictor adăugat, chiar și unul fără legătură; R² ajustat crește doar dacă predictorul aduce mai mult decât ar aduce întâmplarea. Se folosește la compararea modelelor cu număr diferit de predictori.
- `F-statistic: 97.52`, `Prob (F-statistic): 2.88e-42`: testul ipotezei că niciun predictor nu contează (toți coeficienții sunt 0). Valoarea p e practic 0: modelul în ansamblu explică ceva.
- `Log-Likelihood`, `AIC: 192.2`, `BIC: 206.5`: măsuri pentru compararea modelelor pe **aceleași** date; valoarea mai mică e mai bună. Spre deosebire de R², penalizează fiecare predictor în plus (BIC mai tare decât AIC).

**Mijloc: coeficienții.** Pe fiecare rând: `coef` (estimarea), `std err` (eroarea standard: cât de precisă este estimarea), `t` (coef împărțit la eroarea standard), `P>|t|` (valoarea p, ca în C6), `[0.025 0.975]` (intervalul de încredere de 95%).

| | coef | std err | t | P>\|t\| | interval 95% | Citirea |
|---|---|---|---|---|---|---|
| Intercept | 5,42 | 0,33 | 16,6 | 0,000 | 4,77 … 6,06 | log SCI prezis la 1 km și o rată de 1 la mie: o extrapolare, fără sens direct |
| log_distanta | −0,30 | 0,10 | −2,9 | 0,004 | −0,51 … −0,10 | semnificativ: SCI scade cu distanța, la aceeași migrație și graniță |
| log_la_mie | 0,44 | 0,03 | 13,7 | 0,000 | 0,38 … 0,51 | cel mai puternic și cel mai precis: t de 13,7 |
| granita | 0,11 | 0,09 | 1,3 | 0,210 | −0,06 … 0,28 | nesemnificativ: p peste 0,05, intervalul conține 0 |

Regula practică: un t peste 2 în valoare absolută corespunde unei valori p sub 0,05.

**Jos: diagnosticul reziduurilor.** Modelul liniar presupune reziduuri aproximativ normale, independente, cu aceeași împrăștiere.
- `Omnibus: 19.8` (p < 0,001) și `Jarque-Bera: 22.2` (p < 0,001): reziduurile nu sunt perfect normale. `Skew: 0.65`: coadă spre dreapta, adică câteva perechi mult peste model (Islanda). `Kurtosis: 3.57`: puțin peste 3, valoarea distribuției normale. Cu 263 de perechi, abaterea moderată de la normalitate nu schimbă mult coeficienții, dar trebuie spusă.
- `Durbin-Watson: 2.02`: aproape de 2, fără autocorelație între rândurile vecine. Contează mai ales la serii de timp; aici ordinea rândurilor este arbitrară.
- `Cond. No.: 54`: numărul de condiționare; valorile mari (statsmodels avertizează peste 1.000) semnalează multicoliniaritate sau scări foarte diferite. Aici e mic, ca VIF-urile din secțiunea 4.
- Nota de sub tabel („Standard Errors assume that the covariance matrix of the errors is correctly specified”) amintește că erorile standard presupun observații independente. Perechile nu sunt independente (Germania apare în 23 de perechi), deci erorile standard sunt subestimate; corecția uzuală este gruparea pe țară.

**Exercițiu.** R² ajustat al lui M3 și al lui M4 (`rsquared_adj`). Care e mai mare? Ce înseamnă pentru granița adăugată în M4?

In [ ]:
# Atributul rsquared_adj al lui m3 și al lui m4, separate prin virgulă

### 6.2. Coeficienții, în cuvinte
Când ambele părți sunt pe logaritm, coeficientul este o **elasticitate**: dacă predictorul se înmulțește cu 2, SCI se înmulțește cu 2 la puterea coeficientului. Pentru granița (0 sau 1), SCI se înmulțește cu 10 la puterea coeficientului.

In [ ]:
b = m4.params
print("distanța de două ori mai mare:   SCI ×", round(2 ** b["log_distanta"], 2))
print("rata migranților de două ori:    SCI ×", round(2 ** b["log_la_mie"], 2))
print("graniță comună:                  SCI ×", round(10 ** b["granita"], 2))

O distanță de două ori mai mare merge cu un SCI cu aproximativ 19% mai mic; o rată a migranților de două ori mai mare, cu un SCI cu aproximativ 36% mai mare; țările vecine au un SCI cu aproximativ 29% mai mare, la aceeași distanță și migrație, dar intervalul graniței conține 0.

### 6.3. Interpretarea, ca într-un raport
Pe 263 de perechi de țări europene, un model liniar cu distanța, rata migranților și granița comună explică 53% din variația lui log SCI (R² ajustat 0,52). **Migrația este predictorul cel mai puternic**: la aceeași distanță și graniță, o rată a migranților de două ori mai mare merge cu un SCI cu aproximativ 36% mai mare (coeficient 0,44, interval de 95% între 0,38 și 0,51). **Distanța rămâne semnificativă, dar mult mai slabă decât părea singură**: coeficientul ei scade de la −0,76 în M1 la −0,30 în M4, pentru că o parte din efectul ei trecea prin migrație. **Granița comună nu adaugă un efect clar** odată ce distanța și migrația sunt în model (p = 0,21). Rezultatele sunt asocieri între perechi de țări, nu efecte cauzale; reziduurile nu sunt perfect normale, iar erorile standard sunt subestimate, pentru că aceeași țară apare în multe perechi.

## 7. Reziduurile: peste și sub model
**Reziduul** = log SCI observat − log SCI prezis de M4. Pozitiv: perechea e mai legată decât prezic distanța, migrația și granița. Negativ: mai puțin legată. 10 la puterea reziduului spune de câte ori.

In [ ]:
date["prezis"] = m4.fittedvalues
date["reziduu"] = m4.resid

ax = sns.scatterplot(date, x="prezis", y="log_sci")
ax.axline((4, 4), slope=1, color="black")                 # pe linie: exact cât prezice modelul
ax.set(xlabel="log SCI prezis de M4", ylabel="log SCI observat");

In [ ]:
coloane = ["tara_1", "tara_2", "sci", "la_mie", "distanta_km", "reziduu"]
date.sort_values("reziduu")[coloane].head(5)              # cele mai mult sub model

In [ ]:
date.sort_values("reziduu")[coloane].tail(5)              # cele mai mult peste model

Ghicitoarea din C1, închisă: partenerii României, față de model.

In [ ]:
cu_ro = date[(date["tara_1"] == "RO") | (date["tara_2"] == "RO")].copy()
cu_ro["partener"] = np.where(cu_ro["tara_1"] == "RO", cu_ro["tara_2"], cu_ro["tara_1"])
cu_ro["de_cate_ori"] = 10 ** cu_ro["reziduu"]              # 1,5 = cu 50% peste model
cu_ro.sort_values("reziduu", ascending=False)[["partener", "sci", "la_mie", "reziduu", "de_cate_ori"]].round(2)

In [ ]:
contur["reziduu"] = cu_ro.set_index("partener")["reziduu"]      # reziduul fiecărui partener
ax = contur.plot(column="reziduu", cmap="RdBu", vmin=-0.5, vmax=0.5, legend=True, figsize=(9, 7),
                 missing_kwds={"color": "#eeeeee"})
ax.set(xlim=(2.5e6, 7.4e6), ylim=(1.4e6, 5.5e6), title="Partenerii României: peste (albastru) și sub (roșu) model")
ax.set_axis_off()

Sub model sunt mai ales perechile cu Turcia; peste model, perechile cu Islanda. Partenerii României: Islanda, Luxemburg, Irlanda, Ungaria și Austria sunt peste model; Italia este sub model (un SCI cu aproximativ 30% mai mic decât prezice modelul), deși are cei mai mulți migranți români. Pentru Ungaria, explicația probabilă este minoritatea maghiară din România, pe care modelul nu o include. Islanda și Luxemburgul sunt țări mici: un numitor mic, ca Svalbard în C6.

**Exercițiu.** Partenerii Turciei (`"TR"`), de la cel mai mult sub model la cel mai mult peste. Se poate face apoi aceeași hartă ca pentru România.

In [ ]:
# 1. Perechile în care tara_1 sau tara_2 este "TR" (două condiții, cu |), ca pentru România
# 2. Ordonate după reziduu (sort_values), cu lista coloane

## 8. Ce se poate concluziona
- **Asociere, nu cauză.** Modelul arată că SCI merge împreună cu migrația, la aceeași distanță și graniță. Nu arată că migrația produce legăturile: legăturile pot duce și la migrație, iar ambele pot avea o cauză comună (limba, istoria).
- **Nivelul de analiză.** Rezultatele sunt despre perechi de țări, nu despre persoane.
- **Reziduurile sunt întrebări.** Un reziduu mare arată un factor pe care modelul nu îl include: o minoritate, o limbă comună, o particularitate a măsurării (țările mici, cu numitor mic).

**De încercat.** Islanda și Luxemburgul au reziduuri pozitive mari cu multe țări. Care explicație se poate verifica cu datele: o proprietate a oamenilor sau o proprietate a măsurii (SCI împărțit la un număr mic de utilizatori, ca Svalbard în C6)?

In [ ]:
date.to_sql("model_tari", engine, if_exists="replace", index=False)    # tabelul modelului, pentru C8

## Deciziile, de la C6 la model
| Din C6 | În model |
|---|---|
| SCI: asimetrie pozitivă | log SCI |
| Svalbard: numitor foarte mic | exclus |
| distanța și granița, legate | două variabile separate: log distanță și graniță |
| migrația contează ca rată | log rata la mie, nu numărul |
| partenerii și comunitățile urmează migrația | reziduurile arată ce rămâne după migrație |